In [1]:
import pandas as pd
import os
import glob
from collections import defaultdict
import numpy as np
from Bio import SeqIO

In [2]:
## define path
# output directory
path_out = "/Users/kyokokurihara/Lab/projects/2507blastx/output/250909_4474_samples/final_test/figS5C/"

# input files
# sample info table
sradata_path = "/Users/kyokokurihara/Lab/projects/2507blastx/data/Infection_Prediction_Stacking_0_sample_list_TN1000.txt"
# blastx result
blastx_res_path = "/Users/kyokokurihara/Lab/projects/2507blastx/output/250909_4474_samples/taxonomy_output/"
# viral contigs (blacklist virus removed)
blastx_virus_hit_path = "/Users/kyokokurihara/Lab/projects/2507blastx/output/250909_4474_samples/virus_contig_path.txt"
# all fasta files
path_in_fasta = "/Users/kyokokurihara/Lab/projects/2507blastx/data/250819_4474samples/contig_viral_hit/"
# virus blasklist
blacklist_path = "/Users/kyokokurihara/Lab/projects/2507blastx/data/black_list_TN_ISG_low_new_format.txt"
# blastn result (blacklist virus removed)
blastn_input = "/Users/kyokokurihara/Lab/projects/2507blastx/output/250909_4474_samples/blastn_result_top_hit.tsv"
# blastn result (blacklist virus)
path_in_blastn = path_out + "virus_hit_all_contigs_w_bl.txt"

if not os.path.exists(path_out):
    os.mkdir(path_out)
print("saving files:", path_out)

saving files: /Users/kyokokurihara/Lab/projects/2507blastx/output/250909_4474_samples/final_test/figS5C/


# Blastx tophit without blacklist virus removal

In [3]:
## blastx tophit w/o blacklist filter
# sample info table
sampledf = pd.read_table(sradata_path).set_index("ID")

# read blastx outputs
paths = sorted(glob.glob(blastx_res_path + "*.virus.txt"))
print("# all files:", len(paths))

dfs = []
num_emp = 0  ##
for fp in paths:
    # read blastx tophit
    df = pd.read_csv(fp, sep="\t")
    # check NaN
    if df.empty:
        num_emp += 1  ##
        continue
    if df.isna().sum().sum() != 0:
        print(f"{fp} has empty cells")
    # add sample ID from query id
    df["sample"] = df["qseqid"].apply(lambda x: x.split("_")[0])
    # add cm from sample info table
    df["cm"] = df["sample"].apply(lambda x: sampledf.loc[x, "CM"])
    dfs.append(df)

# concat dfs
df_all = pd.concat(dfs, ignore_index=True)

print("# samples wo virus hit", num_emp)
print("# samples with virus hit:", len(set(df_all["sample"].to_list())))
print("blastx tophit table size:", df_all.shape[0])
print("#NaN in df_all", df_all.isna().sum().sum())

# save
df_all.to_csv(path_out + "virus_all_wo_black_filter.tsv", sep="\t")
# show
df_all.head()

# all files: 4474
# samples wo virus hit 309
# samples with virus hit: 4165
blastx tophit table size: 48901
#NaN in df_all 0


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,staxids_lst,name_lst,group_lst,taxid_blastx,family,genus,species,is_virus,sample,cm
0,ERR11505147_21643,QDK64674.1,64.5,355,126,0,711,1775,226,580,...,['2594425'],['Retroviridae;Betaretrovirus;Black Syrian ham...,['unclassified Viruses domain'],2594425,Retroviridae,Betaretrovirus,Black Syrian hamster retrovirus,1,ERR11505147,FN
1,ERR11505147_3806,P00532.1,100.0,113,0,0,1,339,45,157,...,['11812'],['Retroviridae;Gammaretrovirus;Gammaretrovirus...,['unclassified Viruses domain'],11812,Retroviridae,Gammaretrovirus,Gammaretrovirus momursar,1,ERR11505147,FN
2,ERR11505147_42889,AAA42545.1,98.0,498,10,0,2120,627,266,763,...,['12438'],['Retroviridae;unclassified Retroviridae genus...,['unclassified Viruses domain'],12438,Retroviridae,unclassified Retroviridae genus,AKT8 retrovirus,1,ERR11505147,FN
3,ERR11505147_43931,QDK64674.1,47.6,191,99,1,3,575,589,778,...,['2594425'],['Retroviridae;Betaretrovirus;Black Syrian ham...,['unclassified Viruses domain'],2594425,Retroviridae,Betaretrovirus,Black Syrian hamster retrovirus,1,ERR11505147,FN
4,ERR11505147_77878,URX65451.1,96.3,54,2,0,163,2,93,146,...,['11768'],['Retroviridae;Gammaretrovirus;Gammaretrovirus...,['unclassified Viruses domain'],11768,Retroviridae,Gammaretrovirus,Gammaretrovirus felleu,1,ERR11505147,FN


# BLASTn

In [4]:
## create fasta file of viral contigs
def collect_from_tsv(tsv_file, output_file):
    """
    Read fasta path + contig IDs from a TSV and write selected sequences
    into a single multi-FASTA file.

    Parameters:
        tsv_file (str): Path to TSV file with columns [path, qseqid]
        output_file (str): Path to output fasta file
    """
    df = pd.read_table(tsv_file, sep="\t")
    with open(output_file, "w") as out_f:
        for fasta_file in sorted(list(set(df["path"].to_list()))):
            record_dict = SeqIO.index(fasta_file, "fasta")
            contigs = df.loc[df["path"] == fasta_file, "qseqid"].to_list()
            for contig in contigs:
                if contig in record_dict:
                    SeqIO.write(record_dict[contig], out_f, "fasta")
                else:
                    print(f"[Warning] {contig} not found in {fasta_file}")
    print("done")


# remove qseqs already analyzed
qseqs_done = sorted(list(set(pd.read_table(blastx_virus_hit_path)["qseqid"].to_list())))
print("# num qseqs done:", len(qseqs_done))
df_slim = df_all.loc[~(df_all["qseqid"].isin(qseqs_done))].copy()
print("# num qseqs todo:", len(sorted(list(set(df_slim["qseqid"].to_list())))))

# get fasta file path
for idx in df_slim.index:
    df_slim.loc[idx, "path"] = f"{path_in_fasta}{df_slim.loc[idx, "cm"]}/{df_slim.loc[idx, "sample"]}.viral_hit_contigs.fa"
df_fasta = df_slim[["path", "qseqid"]].copy()
df_fasta.to_csv(path_out + "virus_contig_path_w_bl.txt", sep="\t", index=False)

# show
print(df_fasta.head(3))

# gather fasta files of viral hit
collect_from_tsv(path_out + "virus_contig_path_w_bl.txt", path_out + "virus_hit_all_contigs_w_bl.fasta")

# num qseqs done: 14353
# num qseqs todo: 34548
                                                path             qseqid
0  /Users/kyokokurihara/Lab/projects/2507blastx/d...  ERR11505147_21643
1  /Users/kyokokurihara/Lab/projects/2507blastx/d...   ERR11505147_3806
2  /Users/kyokokurihara/Lab/projects/2507blastx/d...  ERR11505147_42889
done


Run BLASTn (script: `FigS5C_blastn.sh`, output file name: `path_in_blastn`)

In [5]:
## taxonkit
# path for taxonkit result
takonkit_path_2 = path_out + "taxids_blastn.txt"

# initialize set
taxids = set()

# read blastn result
col_name= [
    "qseqid", "sseqid", "pident", "length", 
    "mismatch", "gapopen", "qstart", "qend", 
    "sstart", "send", "evalue", "bitscore", 
    "qframe", "staxids", "stitle", "qlen", 
    "slen", "qcovs", "qcovhsp"
]

blastn = pd.read_table(path_in_blastn, header=None, dtype={13:str})  # staxids as string
blastn.columns = col_name
print("blastn output original size:", blastn.shape)
print("#NaN:", blastn.isna().sum().sum())

# sort by bitscore
blastn_sorted = blastn.sort_values("bitscore", ascending=False)
# delete duplicates
blastn_uniq = blastn_sorted.groupby("qseqid").first()
print("blastn output size after choosing tophits:", blastn_uniq.shape)

# get taxid
for line_taxid in blastn_uniq["staxids"].to_list():
    if (line_taxid is None) or (line_taxid!=line_taxid):
        continue
    raw_taxids = line_taxid.split(";")
    for i in range(len(raw_taxids)):
        taxids.add(raw_taxids[i])

# write taxid per line
with open(takonkit_path_2, "w") as f:
    for taxid in taxids:
        f.write(taxid + "\n")

print(f"{len(taxids)} taxids in total")

# show
blastn_uniq.head(10)

blastn output original size: (4700369, 19)
#NaN: 0
blastn output size after choosing tophits: (30967, 18)
621 taxids in total


,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qframe,staxids,stitle,qlen,slen,qcovs,qcovhsp
qseqid,,,,,,,,,,,,,,,,,,
DRR100946_24444,gi|191391|gb|M10134.1|HAMIAP18C,99.752,805,2,0,1,805,3186,3990,0.000000e+00,1476.0,1,10036,Syrian hamster intracisternal A particle (IAP)...,805,7951,100,100
DRR100946_32054,gi|2025820552|ref|XM_040741093.1|,92.405,79,6,0,298,376,1369,1447,2.460000e-20,113.0,1,10036,PREDICTED: Mesocricetus auratus speckle-type P...,530,1601,15,15
DRR100946_6903,gi|191391|gb|M10134.1|HAMIAP18C,99.453,1827,6,4,1,1826,4009,5832,0.000000e+00,3315.0,1,10036,Syrian hamster intracisternal A particle (IAP)...,1826,7951,100,100
DRR169424_11093,gi|19401498|gb|AF417223.1|,100.000,461,0,0,1,461,1560,2020,0.000000e+00,852.0,1,509154,"Pig endogenous retrovirus clone CCa2 env gene,...",461,2219,100,100
DRR169424_111404,gi|21667840|emb|AL772224.2|,99.265,272,2,0,1,272,10514,10785,1.300000e-134,492.0,1,10090,Mouse DNA sequence from clone RP23-131N18 on c...,272,182544,100,100
DRR169424_120590,gi|39777306|gb|AY468414.1|,99.153,236,2,0,1,236,346,111,1.130000e-114,425.0,1,10566,Human papillomavirus isolate FA119 major capsi...,236,443,100,100
DRR169424_161479,gi|288779261|dbj|AK349086.1|,100.000,296,0,0,1,296,202,497,3.010000e-151,547.0,1,9823,"Sus scrofa mRNA, clone:PCT010024A02, expressed...",296,1824,100,100
DRR169424_168821,gi|55909010|gb|AC140230.3|,100.000,680,0,0,1,680,27997,28676,0.000000e+00,1256.0,1,10090,"Mus musculus BAC clone RP24-493E4 from 14, com...",680,154085,100,100
DRR169424_258240,gi|2620235828|gb|OR777213.1|,100.000,229,0,0,3,231,12579,12351,3.980000e-114,424.0,1,10515,Human adenovirus 2 isolate HAdV-C2/USA/13G6/20...,231,35920,99,99


Run taxonkit (`FigS5C_taxonkit.sh`) in `path_out` directory.

In [6]:
## phage list
# read taxonkit output
taxinfo = pd.read_table(path_out + "taxids_long_blastn.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# add taxonomy - {d};{K};{p};{c};{o};{f};{g};{s}
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# count NaN
print("#NaN:", taxinfo.isna().sum().sum())
# fill NaN
taxinfo = taxinfo.fillna("-")
# size
print("taxonkit result size:", taxinfo.shape[0])

# find Caudoviricetes, phage, Inoviridae, Lavidaviridae, Microviridae
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    lineage = taxinfo.loc[idx, "lineage"]
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. phage sp
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# phage group order
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# size
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# save table
taxinfo.to_csv(path_out + "taxids_phage.tsv", sep="\t")

# save phage list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list_blastn.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# show
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 0
taxonkit result size: 621
phage order: []
Caudoviricetes: 0
phage: 0
 * not Caudoviricetes: 0
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae


In [7]:
## add lineage info to taxid list
# input path
taxid_in = path_out + "taxids_lineage_blastn.txt"

# read lineage table
lineage_info = pd.read_table(taxid_in, header=None, dtype={0:str})
lineage_info.columns = ["taxid", "domain", "fgs"]

# check NaN
print("#NaN:", lineage_info.isna().sum().sum())

# add family, genus, speceis columns
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)


# replace domain with phage for phage group
phage_ids = []
with open(path_out + "phage_list_blastn.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# fill NaN
lineage_info = lineage_info.fillna("-")

# size
print(f"{lineage_info.shape[0]} taxids in total\n")

# show example hit for each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# show
lineage_info

#NaN: 0
621 taxids in total

All domain: {'Bacteria', 'unclassified Viruses domain', 'Eukaryota', 'unclassified other entries domain'} 

##### Bacteria ####
size: 4
          domain                   species
taxid                                     
2100    Bacteria  Mesomycoplasma hyorhinis
1404    Bacteria       Priestia megaterium
224308  Bacteria         Bacillus subtilis

##### unclassified Viruses domain ####
size: 260
                              domain                 species
taxid                                                       
1504288  unclassified Viruses domain  Macavirus bovinegamma6
141903   unclassified Viruses domain      Rat retrovirus SC1
400122   unclassified Viruses domain        Circovirus finch

##### Eukaryota ####
size: 234
          domain                species
taxid                                  
27606  Eukaryota    Eubalaena glacialis
9669   Eukaryota       Mustela putorius
10047  Eukaryota  Meriones unguiculatus

##### unclassified other entries

,domain,fgs,family,genus,species
taxid,,,,,
1504288,unclassified Viruses domain,Orthoherpesviridae;Macavirus;Macavirus bovineg...,Orthoherpesviridae,Macavirus,Macavirus bovinegamma6
141903,unclassified Viruses domain,Retroviridae;unclassified Retroviridae genus;R...,Retroviridae,unclassified Retroviridae genus,Rat retrovirus SC1
2871168,unclassified other entries domain,unclassified other entries family;unclassified...,unclassified other entries family,unclassified other entries genus,Lentiviral expression vector MCPyV
27606,Eukaryota,Balaenidae;Eubalaena;Eubalaena glacialis,Balaenidae,Eubalaena,Eubalaena glacialis
400122,unclassified Viruses domain,Circoviridae;Circovirus;Circovirus finch,Circoviridae,Circovirus,Circovirus finch
...,...,...,...,...,...
1513256,unclassified Viruses domain,Papillomaviridae;Gammapapillomavirus;Gammapapi...,Papillomaviridae,Gammapapillomavirus,Gammapapillomavirus 11
10376,unclassified Viruses domain,Orthoherpesviridae;Lymphocryptovirus;Lymphocry...,Orthoherpesviridae,Lymphocryptovirus,Lymphocryptovirus humangamma4
679108,unclassified other entries domain,unclassified other entries family;unclassified...,unclassified other entries family,unclassified other entries genus,EIAV-based lentiviral vector


In [8]:
## merge blastn result and taxomoy info
# read blastn result
blastn_tax = blastn_uniq.copy()
blastn_tax["staxids_lst"] = blastn_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
blastn_tax["name_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
blastn_tax["group_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

# add taxonomy lineage
num_ambiguous_hit = 0
for idx in blastn_uniq.index:
    groups = blastn_tax.loc[idx, "group_lst"]
    names = blastn_tax.loc[idx, "name_lst"]
    # count hits with taxids with multiple groups
    if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
        num_ambiguous_hit += 1
        print("#### Different root ####")
        print(groups)
        print(names)
        print()
print("Num ambiguou hits:", num_ambiguous_hit)

# add family, genus, species
blastn_tax["taxid"] = blastn_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  # added on 2025/09/01
blastn_tax["family"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "family"] if((x is not None) and (x==x)) else x)
blastn_tax["genus"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "genus"] if((x is not None) and (x==x)) else x)
blastn_tax["species"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "species"] if((x is not None) and (x==x)) else x)

# check NaN
print("#NaN:", blastn_tax.isna().sum().sum())

# add tag if list has virus - virus = yes when all classification are virus
blastn_tax["is_virus"] = blastn_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

# edit columns name
blastn_tax.columns = blastn_tax.columns.map(lambda x: x + "_blastn")  # add "_blastn" to column names

# save
blastn_tax.to_csv(path_out + "blastn_result_top_hit.tsv", sep="\t")

# show
print("Size after adding taxonomy:", blastn_tax.shape)
blastn_tax.head(10)

#### Different root ####
['Eukaryota', 'unclassified Viruses domain']
['Hominidae;Homo;Homo sapiens', 'Retroviridae;unclassified Retroviridae genus;Human endogenous retrovirus K']

Num ambiguou hits: 1
#NaN: 0
Size after adding taxonomy: (30967, 26)


,sseqid_blastn,pident_blastn,length_blastn,mismatch_blastn,gapopen_blastn,qstart_blastn,qend_blastn,sstart_blastn,send_blastn,evalue_blastn,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
qseqid,,,,,,,,,,,,,,,,,,,,,
DRR100946_24444,gi|191391|gb|M10134.1|HAMIAP18C,99.752,805,2,0,1,805,3186,3990,0.000000e+00,...,100,100,[10036],[Cricetidae;Mesocricetus;Mesocricetus auratus],[Eukaryota],10036,Cricetidae,Mesocricetus,Mesocricetus auratus,0
DRR100946_32054,gi|2025820552|ref|XM_040741093.1|,92.405,79,6,0,298,376,1369,1447,2.460000e-20,...,15,15,[10036],[Cricetidae;Mesocricetus;Mesocricetus auratus],[Eukaryota],10036,Cricetidae,Mesocricetus,Mesocricetus auratus,0
DRR100946_6903,gi|191391|gb|M10134.1|HAMIAP18C,99.453,1827,6,4,1,1826,4009,5832,0.000000e+00,...,100,100,[10036],[Cricetidae;Mesocricetus;Mesocricetus auratus],[Eukaryota],10036,Cricetidae,Mesocricetus,Mesocricetus auratus,0
DRR169424_11093,gi|19401498|gb|AF417223.1|,100.000,461,0,0,1,461,1560,2020,0.000000e+00,...,100,100,[509154],[Retroviridae;Gammaretrovirus;Gammaretrovirus ...,[unclassified Viruses domain],509154,Retroviridae,Gammaretrovirus,Gammaretrovirus porConc,1
DRR169424_111404,gi|21667840|emb|AL772224.2|,99.265,272,2,0,1,272,10514,10785,1.300000e-134,...,100,100,[10090],[Muridae;Mus;Mus musculus],[Eukaryota],10090,Muridae,Mus,Mus musculus,0
DRR169424_120590,gi|39777306|gb|AY468414.1|,99.153,236,2,0,1,236,346,111,1.130000e-114,...,100,100,[10566],[Papillomaviridae;unclassified Papillomavirida...,[unclassified Viruses domain],10566,Papillomaviridae,unclassified Papillomaviridae genus,Human papillomavirus,1
DRR169424_161479,gi|288779261|dbj|AK349086.1|,100.000,296,0,0,1,296,202,497,3.010000e-151,...,100,100,[9823],[Suidae;Sus;Sus scrofa],[Eukaryota],9823,Suidae,Sus,Sus scrofa,0
DRR169424_168821,gi|55909010|gb|AC140230.3|,100.000,680,0,0,1,680,27997,28676,0.000000e+00,...,100,100,[10090],[Muridae;Mus;Mus musculus],[Eukaryota],10090,Muridae,Mus,Mus musculus,0
DRR169424_258240,gi|2620235828|gb|OR777213.1|,100.000,229,0,0,3,231,12579,12351,3.980000e-114,...,99,99,[10515],[Adenoviridae;Mastadenovirus;Human mastadenovi...,[unclassified Viruses domain],10515,Adenoviridae,Mastadenovirus,Human mastadenovirus C,1


In [9]:
# show hits with multiple tax ids
blastn_tax_double = blastn_tax.loc[blastn_tax["staxids_lst_blastn"].apply(lambda x: True if len(x)>1 else False), :].groupby("staxids_blastn", as_index=False).first()
blastn_tax_double.head(10)

,staxids_blastn,sseqid_blastn,pident_blastn,length_blastn,mismatch_blastn,gapopen_blastn,qstart_blastn,qend_blastn,sstart_blastn,send_blastn,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
0,108098;129951,gi|2618371706|emb|OY757741.1|,99.003,301,1,2,1,300,15055,14756,...,90,90,"[108098, 129951]",[Adenoviridae;Mastadenovirus;Human mastadenovi...,"[unclassified Viruses domain, unclassified Vir...",108098,Adenoviridae,Mastadenovirus,Human mastadenovirus B,1
1,11096;149596,gi|12657941|ref|NC_002657.1|,97.351,151,4,0,1,151,2063,2213,...,100,100,"[11096, 149596]","[Flaviviridae;Pestivirus;Pestivirus suis, Flav...","[unclassified Viruses domain, unclassified Vir...",11096,Flaviviridae,Pestivirus,Pestivirus suis,1
2,11856;11857,gi|9627014|ref|NC_001514.1|,99.710,345,1,0,1,345,4567,4223,...,100,100,"[11856, 11857]",[Retroviridae;Betaretrovirus;Betaretrovirus sq...,"[unclassified Viruses domain, unclassified Vir...",11856,Retroviridae,Betaretrovirus,Betaretrovirus squmon,1
3,123358;1891746,gi|37499133|gb|AY327113.1|,98.844,173,2,0,1,173,880,1052,...,100,100,"[123358, 1891746]",[Polyomaviridae;Gammapolyomavirus;Gammapolyoma...,"[unclassified Viruses domain, unclassified Vir...",123358,Polyomaviridae,Gammapolyomavirus,Gammapolyomavirus anseris,1
4,1689785;1914447,gi|1464310537|ref|NC_038964.1|,92.667,300,22,0,1,300,8634,8933,...,100,100,"[1689785, 1914447]","[Flaviviridae;Pestivirus;Pestivirus scrofae, F...","[unclassified Viruses domain, unclassified Vir...",1689785,Flaviviridae,Pestivirus,Pestivirus scrofae,1
5,199308;1960251,gi|2047106742|ref|NC_055234.1|,100.000,104,0,0,40,143,57848,57951,...,57,42,"[199308, 1960251]",[Orthoherpesviridae;Macavirus;Macavirus suidga...,"[unclassified Viruses domain, unclassified Vir...",199308,Orthoherpesviridae,Macavirus,Macavirus suidgamma5,1
6,272551;2560570,gi|66476549|ref|NC_007016.1|,79.787,94,15,4,139,230,48671,48762,...,32,32,"[272551, 2560570]",[Orthoherpesviridae;Rhadinovirus;Rhadinovirus ...,"[unclassified Viruses domain, unclassified Vir...",272551,Orthoherpesviridae,Rhadinovirus,Rhadinovirus macacinegamma11,1
7,673351;673352,gi|320542831|gb|HQ456169.1|,98.707,232,3,0,1,232,410,179,...,100,100,"[673351, 673352]","[Suidae;Sus;Sus barbatus, Suidae;Sus;Sus barba...","[Eukaryota, Eukaryota]",673351,Suidae,Sus,Sus barbatus,0
8,91740;1960249,gi|1464306536|ref|NC_038264.1|,100.000,268,0,0,1,268,13942,14209,...,100,100,"[91740, 1960249]",[Orthoherpesviridae;Macavirus;Macavirus suidga...,"[unclassified Viruses domain, unclassified Vir...",91740,Orthoherpesviridae,Macavirus,Macavirus suidgamma3,1
9,91741;1960250,gi|1464306584|ref|NC_038265.1|,100.000,151,0,0,1,151,4205,4355,...,100,100,"[91741, 1960250]",[Orthoherpesviridae;Macavirus;Macavirus suidga...,"[unclassified Viruses domain, unclassified Vir...",91741,Orthoherpesviridae,Macavirus,Macavirus suidgamma4,1


In [10]:
## merge blastn result and blastx result
# read blastx tables
df_all = pd.read_table(path_out + "virus_all_wo_black_filter.tsv", index_col=0)
# remove Retroviridae
df_no_retro_all = df_all[df_all["family"]!="Retroviridae"].copy()

# add blastn
# blastn of blacklist virus removed
df_blastn_prev = pd.read_table(blastn_input)
print("df_blastn_prev shape:", df_blastn_prev.shape)
# blast n of blacklist virus
df_blastn_add = pd.read_table(path_out + "blastn_result_top_hit.tsv")
print("df_blastn_add shape:", df_blastn_add.shape)
# merge
df_blastn = pd.concat([df_blastn_prev, df_blastn_add])
print("df_blastn shape:", df_blastn.shape)

# merge blastn and blastx
df_all_blastn_no_retro = df_no_retro_all.merge(df_blastn, on="qseqid", how="left")
print("df_all_blastn_no_retro shape:", df_all_blastn_no_retro.shape)
print("#NaN", df_all_blastn_no_retro.isna().sum().sum())

# save
df_all_blastn_no_retro.to_csv(path_out + "virus_all_wo_black_filter_no_retro.blastn.tsv", sep="\t")
# show
df_all_blastn_no_retro

df_blastn_prev shape: (11981, 27)
df_blastn_add shape: (30967, 27)
df_blastn shape: (42948, 27)
df_all_blastn_no_retro shape: (17083, 55)
#NaN 68458


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
0,ERR11505147_9442,AWC68490.1,50.0,96,47,1,60,344,7,102,...,100.0,100.0,['38674'],['Cricetidae;Onychomys;Onychomys torridus'],['Eukaryota'],38674.0,Cricetidae,Onychomys,Onychomys torridus,0.0
1,ERR1682084_141504,QYV42621.1,100.0,313,0,0,3,941,55,367,...,100.0,100.0,['2809445'],['Anelloviridae;Gyrovirus;Gyrovirus sp.'],['unclassified Viruses domain'],2809445.0,Anelloviridae,Gyrovirus,Gyrovirus sp.,1.0
2,ERR1682084_266934,AFJ92648.1,100.0,140,0,0,420,1,92,231,...,100.0,100.0,['1002273'],['Anelloviridae;Gyrovirus;Gyrovirus galga1'],['unclassified Viruses domain'],1002273.0,Anelloviridae,Gyrovirus,Gyrovirus galga1,1.0
3,ERR1682089_180946,YP_010796956.1,59.1,44,18,0,132,1,88,131,...,89.0,89.0,['8845'],['Anatidae;Anser;Anser cygnoides'],['Eukaryota'],8845.0,Anatidae,Anser,Anser cygnoides,0.0
4,ERR1682089_446091,UAU47043.1,99.5,221,1,0,664,2,140,360,...,100.0,100.0,['2583049'],['Anelloviridae;Gyrovirus;Gyrovirus 3'],['unclassified Viruses domain'],2583049.0,Anelloviridae,Gyrovirus,Gyrovirus 3,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17078,SRR988674_5076,AEP83554.1,100.0,206,0,0,1,618,799,1004,...,100.0,100.0,['28344'],['Arteriviridae;Betaarterivirus;Betaarteriviru...,['unclassified Viruses domain'],28344.0,Arteriviridae,Betaarterivirus,Betaarterivirus americense,1.0
17079,SRR988674_7268,YP_009667148.1,100.0,802,0,0,3,2408,662,1463,...,100.0,100.0,"['11049', '1965066']",['Arteriviridae;Betaarterivirus;Betaarteriviru...,"['unclassified Viruses domain', 'unclassified ...",11049.0,Arteriviridae,Betaarterivirus,Betaarterivirus suid 1,1.0
17080,SRR988674_8034,AEP83554.1,100.0,401,0,0,1205,3,1247,1647,...,100.0,100.0,['28344'],['Arteriviridae;Betaarterivirus;Betaarteriviru...,['unclassified Viruses domain'],28344.0,Arteriviridae,Betaarterivirus,Betaarterivirus americense,1.0
17081,SRR988674_8246,YP_009667147.1,99.9,1120,1,0,3378,19,1277,2396,...,99.0,99.0,"['11049', '1965066']",['Arteriviridae;Betaarterivirus;Betaarteriviru...,"['unclassified Viruses domain', 'unclassified ...",11049.0,Arteriviridae,Betaarterivirus,Betaarterivirus suid 1,1.0


# Infection events count

In [11]:
# remove blastn not virus result
df = df_all_blastn_no_retro.copy()
print("original size:", df.shape)
# remove non-viral hit in blastn
df_blastn = df.loc[~(df["is_virus_blastn"]==0), :].copy()
print("size after removeing blastn other hit:", df_blastn.shape)
# show
df_blastn.head(3)

original size: (17083, 55)
size after removeing blastn other hit: (16208, 55)


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
1,ERR1682084_141504,QYV42621.1,100.0,313,0,0,3,941,55,367,...,100.0,100.0,['2809445'],['Anelloviridae;Gyrovirus;Gyrovirus sp.'],['unclassified Viruses domain'],2809445.0,Anelloviridae,Gyrovirus,Gyrovirus sp.,1.0
2,ERR1682084_266934,AFJ92648.1,100.0,140,0,0,420,1,92,231,...,100.0,100.0,['1002273'],['Anelloviridae;Gyrovirus;Gyrovirus galga1'],['unclassified Viruses domain'],1002273.0,Anelloviridae,Gyrovirus,Gyrovirus galga1,1.0
4,ERR1682089_446091,UAU47043.1,99.5,221,1,0,664,2,140,360,...,100.0,100.0,['2583049'],['Anelloviridae;Gyrovirus;Gyrovirus 3'],['unclassified Viruses domain'],2583049.0,Anelloviridae,Gyrovirus,Gyrovirus 3,1.0


In [12]:
## blastn fileter
# total count
count_total= pd.DataFrame(sampledf["CM"].value_counts())
count_total.columns = ["total"]
print("*total")
print(count_total)
print(count_total.sum())

# infection event
df_event = df_blastn[["qseqid", "sample", "family", "genus", "cm"]].groupby(["sample", "genus"]).agg({"qseqid": "count", "cm": "first", "family":"first"}).copy()
df_event.columns = df_event.columns.map(lambda x: "n_contigs" if x=="qseqid" else x)

# infection event per family x cm
df_event = df_event.reset_index()
df_event["sample_genus"] = df_event["sample"] + df_event["genus"]
print("#NaN in df_event:", df_event.isna().sum().sum())
df_family = df_event.reset_index()[["sample_genus", "cm", "family"]].groupby(["family", "cm"]).agg({"sample_genus": "count"}).copy()
# save
df_family.to_csv(path_out + "count_infection_events_including_blacklist_no_retro_per_family_x_cm.blastn.txt", sep="\t")
# show
df_family

*total
    total
CM       
FP   2169
TN   1000
FN    769
TP    536
total    4474
dtype: int64
#NaN in df_event: 0


sample_genus
family                          cm              
Adamaviridae                    FN             3
                                FP             2
Adenoviridae                    FN            18
                                FP            55
                                TN            11
...                                          ...
unclassified Tolivirales family TP            11
unclassified Viruses family     FN            42
                                FP            18
                                TN            10
                                TP            11

[245 rows x 1 columns]